In [ ]:
"""
Uncertainty Analysis for Darcy-Weisbach Permeability Formula
=============================================================
Formula:  k = (h / T) * (mu * L) / (delta_p * A)

Applies the GUM (Guide to the Expression of Uncertainty in Measurement)
law of propagation of uncertainty for a product/quotient formula:

    (δk/k)² = (δh/h)² + (δT/T)² + (δmu/mu)² + (δL/L)² + (δp/δp)² + (δA/A)²
"""

import math

# ── 1. Measured values (SI units) ────────────────────────────────────────────
# The uncertaininty is as 10% FS or +- 5% of the value, whichever is larger.
measurements = {
    "h":     {"value": 10e-3,   "uncertainty": 0.5e-3,  "unit": "m",    "description": "liquid column height"},
    "T":     {"value": 10.0,    "uncertainty": 0.5,     "unit": "s",    "description": "integration time"},
    "mu":    {"value": 6.4e-4,  "uncertainty": 3.2e-5,  "unit": "Pa·s", "description": "fluid viscosity"},
    "L":     {"value": 10e-3,   "uncertainty": 0.5e-3,  "unit": "m",    "description": "sample length"},
    "dp":    {"value": 100e2,   "uncertainty": 500.0,   "unit": "Pa",   "description": "differential pressure (100 mbar)"},
    "A":     {"value": 1e-4,    "uncertainty": 5e-6,    "unit": "m²",   "description": "cross-sectional area"},
}

# ── 2. Calculate k ───────────────────────────────────────────────────────────

v = {name: m["value"] for name, m in measurements.items()}

k = (v["h"] / v["T"]) * (v["mu"] * v["L"]) / (v["dp"] * v["A"])

# ── 3. Relative uncertainties for each variable ──────────────────────────────

rel_uncertainties = {}
for name, m in measurements.items():
    rel = abs(m["uncertainty"] / m["value"])
    rel_uncertainties[name] = rel

# ── 4. Propagate: sum of squares (each variable enters with power ±1) ────────

sum_of_squares = sum(r**2 for r in rel_uncertainties.values())
rel_k = math.sqrt(sum_of_squares)   # relative uncertainty in k
abs_k = k * rel_k                   # absolute uncertainty in k

# ── 5. Contribution of each variable to total variance ───────────────────────

contributions = {
    name: (r**2 / sum_of_squares) * 100
    for name, r in rel_uncertainties.items()
}

# ── 6. Report ────────────────────────────────────────────────────────────────

SEP = "─" * 62

print(SEP)
print("  DARCY-WEISBACH PERMEABILITY — UNCERTAINTY ANALYSIS")
print(SEP)

print("\nMeasured variables:")
print(f"  {'Var':<6} {'Value':>14}  {'± δx':>12}  {'δx/x (%)':>9}  Description")
print(f"  {'─'*6} {'─'*14}  {'─'*12}  {'─'*9}  {'─'*24}")
for name, m in measurements.items():
    rel_pct = rel_uncertainties[name] * 100
    print(f"  {name:<6} {m['value']:>14.4g}  {m['uncertainty']:>12.4g}  {rel_pct:>8.2f}%  {m['description']}")

print(f"\nCalculated permeability:")
print(f"  k  = {k:.3e} m²")
print(f"  δk = {abs_k:.3e} m²")
print(f"  δk/k = {rel_k*100:.2f}%")
print(f"\n  Result: k = ({k:.3e} ± {abs_k:.3e}) m²")
print(f"          k = {k:.3e} m²  [{rel_k*100:.1f}% relative uncertainty]\n")

print("Uncertainty contribution by variable (% of total variance):")
print(f"  {'Var':<6} {'Contribution':>14}  Bar")
print(f"  {'─'*6} {'─'*14}  {'─'*30}")
sorted_contribs = sorted(contributions.items(), key=lambda x: -x[1])
for name, pct in sorted_contribs:
    bar = "█" * int(pct / 2)
    dominant = "  ← dominant" if pct == max(contributions.values()) else ""
    print(f"  {name:<6} {pct:>13.1f}%  {bar}{dominant}")

print(f"\n{SEP}")
print("  GUIDANCE")
print(SEP)
dominant_var = max(contributions, key=contributions.get)
print(f"\n  The dominant uncertainty source is '{dominant_var}' ({contributions[dominant_var]:.1f}% of variance).")
print(f"  Reducing its relative uncertainty would have the largest impact.")
print(f"\n  To reach < 10% total relative uncertainty in k, each variable")
print(f"  must contribute < {(10/math.sqrt(len(measurements))):.1f}% individually (if equally distributed).")
print()